# ConseqaX · Complete GPU benchmark

This is the complete notebook. The matching study bundle and all
launch steps are included. **Do not paste repair snippets or upload a
separate study bundle.**

1. Import this notebook into a **fresh Kaggle session**.
2. Enable **GPU** and **Internet**.
3. Choose **Save Version → Save & Run All once**.

The one executable cell below prepares the study, downloads the pinned
runtime and three model configurations, checks them, and collects the
next scheduled episodes. The seven-hour window includes setup and
leaves time to save outputs inside an eight-hour session.

**Continuing an earlier session?** Before launching, attach the
**entire latest saved output** as an Input, including the active
`conseqax-benchmark` folder, `checkpoint.zip` and supporting files.
The latest raw ledger takes precedence over its older ZIP. Started
slots are never automatically repeated. Keep the entire output after
every session; all **3,456 planned slots** will usually need several
sessions. Never run interactively and then use Save & Run All merely
as a save button, because that launches a new execution; preserve
existing interactive output with Quick Save and attach it next time.

Output streams below and is also retained in `conseqax-diagnostics`.
This notebook includes the missing `limbo` import-path fix. It uses
no paid inference, human evaluation or GitHub Actions. The embedded
archive and every source file are checked against immutable release
hashes before study code runs. Actual GPU compatibility and throughput
still depend on the live session; offline checks are not GPU results.

**Granite format correction:** this version retains Granite's native tool-format instructions alongside the benchmark system message, consistently across all four arms. It is a new frozen template/source condition. Keep the earlier failed qualification output separately; it had zero started episodes and is not a continuation input for this first corrected run. For later sessions, attach only this version's latest output. Before any measured episode, each model must make the required real tool call and return a usable response to the tool result. Exact wording and extra calls remain separate behavior diagnostics.


In [ ]:
import os
import time
# Rerunning a cell must not reset this session's deadline.
if "_SESSION_STARTED_AT" not in globals():
    _SESSION_STARTED_AT = time.monotonic()
os.environ["CONSEQAX_SESSION_START_MONOTONIC"] = str(_SESSION_STARTED_AT)
# Usually leave these two paths alone: a unique attached input is found for you.
BUNDLE_PATH = None       # Optional: delivered ZIP, unpacked study folder, or bundle-manifest.json
RESUME_FROM = None       # Optional: entire previous saved output, active folder, or clean checkpoint.zip
OUTPUT_DIR = "/kaggle/working/conseqax-benchmark"
SESSION_HOURS = 7        # Total study window, including setup; leave hour 8 for saving.
EXPECTED_BUNDLE_SHA256 = 'e87e3ef14cf61ca80676bacbdf98ff6ac2a5d51c66c90588699f280b73af160d'
EXPECTED_SOURCE_COMMIT = 'f5c707cf33e7bfb74785233fdb2f5730fb9ee374'
EXPECTED_MANIFEST_SHA256 = 'd68a1297cf9c39e5290f6c436a15b665b2775ce04dde1d8f821d3167769fcd5e'


from pathlib import Path
import base64
import hashlib

# The small frozen study is inside this notebook, so there is no separate Input
# bundle to upload, select or keep in sync. Its code is verified before execution.
_EMBEDDED_STUDY_BASE64 = (
    '...'
    'Lmpzb25QSwUGAAAAAFUAVQD0IwAATh0JAAAA'
)
_embedded_bytes = base64.b64decode(_EMBEDDED_STUDY_BASE64, validate=True)
if hashlib.sha256(_embedded_bytes).hexdigest() != EXPECTED_BUNDLE_SHA256:
    raise RuntimeError("The embedded study failed its release SHA256 check. No study code was executed.")
_embedded_root = Path(os.environ.get("CONSEQAX_NOTEBOOK_TEMP_ROOT", "/kaggle/temp")) / "conseqax-embedded-study"
_embedded_root.mkdir(parents=True, exist_ok=True)
_embedded_archive = _embedded_root / "conseqax-kaggle-benchmark.zip"
if _embedded_archive.is_symlink():
    raise RuntimeError("Refuse a linked embedded study archive.")
if _embedded_archive.exists():
    if hashlib.sha256(_embedded_archive.read_bytes()).hexdigest() != EXPECTED_BUNDLE_SHA256:
        raise RuntimeError("Cached embedded study bytes changed. Preserve outputs and use a fresh session.")
else:
    with _embedded_archive.open("xb") as _archive_out:
        _archive_out.write(_embedded_bytes)
        _archive_out.flush()
        os.fsync(_archive_out.fileno())
BUNDLE_PATH = str(_embedded_archive)
del _embedded_bytes, _EMBEDDED_STUDY_BASE64
print("Embedded study verified. No separate study Input is required.", flush=True)



from collections import deque
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import re
import shutil
import stat
import subprocess
import sys
import time
import zipfile

# Engineering tests can inspect the plan without GPU, installation, or model calls.
DRY_RUN = os.environ.get("CONSEQAX_NOTEBOOK_DRY_RUN") == "1"
if "_SESSION_STARTED_AT" not in globals():
    _SESSION_STARTED_AT = time.monotonic()
INPUT_ROOT = Path(os.environ.get("CONSEQAX_NOTEBOOK_INPUT_ROOT", "/kaggle/input"))
TEMP_ROOT = Path(os.environ.get("CONSEQAX_NOTEBOOK_TEMP_ROOT", "/kaggle/temp"))
COMMAND_PLAN = []

def require_pins():
    if EXPECTED_BUNDLE_SHA256 is None and EXPECTED_MANIFEST_SHA256 is None:
        raise RuntimeError("Use the released notebook bound to your delivered study: its archive hash is missing, and no unpacked manifest hash is supplied.")
    for label, value in (("archive", EXPECTED_BUNDLE_SHA256), ("manifest", EXPECTED_MANIFEST_SHA256)):
        if value is not None and (not isinstance(value, str) or not re.fullmatch(r"[0-9a-f]{64}", value)):
            raise RuntimeError(f"The released study needs a valid pinned {label} SHA256.")
    if not isinstance(EXPECTED_SOURCE_COMMIT, str) or not re.fullmatch(r"[0-9a-f]{40}", EXPECTED_SOURCE_COMMIT):
        raise RuntimeError("The notebook needs the exact frozen study source commit.")
    if not isinstance(SESSION_HOURS, (int, float)) or isinstance(SESSION_HOURS, bool) or not 0 < SESSION_HOURS <= 7:
        raise RuntimeError("SESSION_HOURS must be greater than zero and at most 7.")

def unique_path(explicit, candidates, label, optional=False):
    if explicit:
        selected = Path(explicit).expanduser().resolve()
        if not selected.exists():
            raise RuntimeError(f"{label} does not exist: {selected}")
        return selected
    choices = sorted({p.resolve() for p in candidates}, key=str)
    if not choices and optional:
        return None
    if len(choices) != 1:
        raise RuntimeError(f"Attach exactly one {label}, or set its path explicitly. Found {len(choices)}.")
    return choices[0]

def discover_bundle():
    if BUNDLE_PATH:
        selected = unique_path(BUNDLE_PATH, [], "study input")
        if selected.is_file() and selected.name == "conseqax-kaggle-benchmark.zip":
            return "zip", selected
        if selected.is_file() and selected.name == "bundle-manifest.json":
            return "tree", selected
        if selected.is_dir():
            manifests = list(selected.rglob("bundle-manifest.json"))
            return "tree", unique_path(None, manifests, "unpacked study manifest")
        raise RuntimeError("Select the delivered ZIP, unpacked study folder, or bundle-manifest.json.")
    archives = list(INPUT_ROOT.rglob("conseqax-kaggle-benchmark.zip"))
    if archives:
        return "zip", unique_path(None, archives, "study ZIP")
    manifests = []
    for candidate in INPUT_ROOT.rglob("bundle-manifest.json"):
        try:
            if candidate.stat().st_size > 4 * 1024 * 1024:
                continue
            record = json.loads(candidate.read_bytes())
            if (isinstance(record, dict)
                    and record.get("schema") == "conseqax-kaggle-source-bundle-v1"
                    and record.get("source_commit") == EXPECTED_SOURCE_COMMIT):
                manifests.append(candidate)
        except (OSError, ValueError, UnicodeError):
            continue
    return "tree", unique_path(None, manifests, "unpacked study manifest matching the pinned commit")

def raw_checkpoint(folder):
    if (folder / "identity.json").is_file():
        if not (folder / "schedule.json").is_file():
            raise RuntimeError("A raw checkpoint is incomplete: identity.json exists but schedule.json is missing. Preserve the whole saved output.")
        return folder.resolve()
    return None

def prefer_raw_checkpoint(selected):
    # A clean export and its active ledger are one campaign. The ledger may retain
    # starts newer than the ZIP; only the runner decides whether its evidence is valid.
    folder = selected if selected.is_dir() else selected.parent
    for candidate in (folder, folder / "conseqax-benchmark"):
        raw = raw_checkpoint(candidate)
        if raw is not None:
            return raw
    if selected.is_dir():
        if (selected / "checkpoint.json").is_file():
            return selected
        if (selected / "checkpoint.zip").is_file():
            return selected / "checkpoint.zip"
        raise RuntimeError("Select the entire saved output, a raw active ledger, or checkpoint.zip.")
    if selected.name != "checkpoint.zip":
        raise RuntimeError("Select the previous output folder or checkpoint.zip.")
    return selected

def discover_resume():
    if RESUME_FROM:
        selected = unique_path(RESUME_FROM, [], "previous checkpoint")
        return prefer_raw_checkpoint(selected)
    roots = set()
    for identity in INPUT_ROOT.rglob("identity.json"):
        roots.add(raw_checkpoint(identity.parent))
    for marker in INPUT_ROOT.rglob("checkpoint.json"):
        roots.add(prefer_raw_checkpoint(marker.parent))
    for archive in INPUT_ROOT.rglob("checkpoint.zip"):
        preferred = prefer_raw_checkpoint(archive)
        if preferred == archive and archive.parent.resolve() in roots:
            continue  # The exported directory and its ZIP are the same snapshot.
        roots.add(preferred)
    return unique_path(None, roots, "previous checkpoint", optional=True)

def open_bundle(archive, destination):
    if EXPECTED_BUNDLE_SHA256 is None:
        raise RuntimeError("ZIP input requires its released archive SHA256 pin.")
    with archive.open("rb") as stream:
        archive_hash = hashlib.file_digest(stream, "sha256").hexdigest()
    if archive_hash != EXPECTED_BUNDLE_SHA256:
        raise RuntimeError("Study ZIP hash differs from the released notebook. No code was executed.")
    if destination.exists():
        raise RuntimeError("Study extraction already exists. Restart the kernel/session rather than replacing frozen files.")
    with zipfile.ZipFile(archive) as source:
        members = source.infolist()
        if len(members) > 20000 or sum(m.file_size for m in members) > 512 * 1024 * 1024:
            raise RuntimeError("Study archive exceeds the notebook's safe extraction limits.")
        seen = set()
        for member in members:
            name = member.filename
            parts = PurePosixPath(name).parts
            if (not parts or parts[0] != "conseqax-kaggle-study" or "\\" in name
                    or PurePosixPath(name).is_absolute() or any(p in ("..", ".") or ":" in p for p in parts)
                    or "\x00" in name or name in seen
                    or str(PurePosixPath(name)) != name.rstrip("/")):
                raise RuntimeError("Unsafe or duplicate study archive member.")
            seen.add(name)
            mode = member.external_attr >> 16
            if stat.S_ISLNK(mode) or (stat.S_IFMT(mode) and not (stat.S_ISREG(mode) or stat.S_ISDIR(mode))):
                raise RuntimeError("Study archive contains a link or special file.")
            if member.flag_bits & 1:
                raise RuntimeError("Encrypted study archive is unsupported.")
        manifest_name = "conseqax-kaggle-study/bundle-manifest.json"
        manifest = json.loads(source.read(manifest_name))
        if manifest.get("schema") != "conseqax-kaggle-source-bundle-v1":
            raise RuntimeError("Unsupported study bundle manifest schema.")
        if manifest.get("source_commit") != EXPECTED_SOURCE_COMMIT:
            raise RuntimeError("Study source commit differs from the released notebook.")
        bindings = manifest.get("files")
        if not isinstance(bindings, dict) or not bindings:
            raise RuntimeError("Study bundle has no pinned file inventory.")
        inventory = {m.filename[len("conseqax-kaggle-study/"):] for m in members if not m.is_dir()}
        if inventory != set(bindings) | {"bundle-manifest.json"}:
            raise RuntimeError("Study archive and manifest inventories differ.")
        for relative, binding in bindings.items():
            data = source.read("conseqax-kaggle-study/" + relative)
            if (not isinstance(binding, dict) or type(binding.get("bytes")) is not int
                    or len(data) != binding["bytes"] or hashlib.sha256(data).hexdigest() != binding.get("sha256")):
                raise RuntimeError(f"Study source binding failed: {relative}")
        destination.mkdir(parents=True, exist_ok=False)
        try:
            for member in members:
                if not member.is_dir():
                    target = destination.joinpath(*PurePosixPath(member.filename).parts)
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with source.open(member) as incoming, target.open("xb") as outgoing:
                        shutil.copyfileobj(incoming, outgoing)
        except BaseException:
            # Preserve the incomplete directory for inspection; never execute it.
            raise
    return destination / "conseqax-kaggle-study"

def open_unpacked_bundle(manifest_path, destination):
    if EXPECTED_MANIFEST_SHA256 is None:
        raise RuntimeError("Unpacked study input requires the released manifest SHA256 pin.")
    tree = manifest_path.parent
    if manifest_path.is_symlink() or tree.is_symlink():
        raise RuntimeError("Unpacked study manifest/root must not be a symbolic link.")
    manifest_bytes = manifest_path.read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != EXPECTED_MANIFEST_SHA256:
        raise RuntimeError("Unpacked study manifest SHA256 differs from the release pin. No code was executed.")
    manifest = json.loads(manifest_bytes)
    if (manifest.get("schema") != "conseqax-kaggle-source-bundle-v1"
            or manifest.get("source_commit") != EXPECTED_SOURCE_COMMIT):
        raise RuntimeError("Unpacked study schema or source commit differs from the released notebook.")
    bindings = manifest.get("files")
    if not isinstance(bindings, dict) or not bindings or len(bindings) > 20000:
        raise RuntimeError("Unpacked study manifest has an invalid file inventory.")
    observed = set()
    for path in tree.rglob("*"):
        if path.is_symlink():
            raise RuntimeError("Unpacked study contains a symbolic link.")
        if not path.is_dir():
            if not stat.S_ISREG(path.stat().st_mode):
                raise RuntimeError("Unpacked study contains a special file.")
            observed.add(path.relative_to(tree).as_posix())
    if observed != set(bindings) | {"bundle-manifest.json"}:
        raise RuntimeError("Unpacked study and manifest inventories differ.")
    payload = {}
    total_bytes = 0
    for relative, binding in bindings.items():
        if not isinstance(relative, str):
            raise RuntimeError("Unsafe unpacked study path.")
        parts = PurePosixPath(relative).parts
        if (not parts or PurePosixPath(relative).is_absolute() or "\\" in relative
                or "\x00" in relative or any(p in ("..", ".") or ":" in p for p in parts)
                or str(PurePosixPath(relative)) != relative):
            raise RuntimeError("Unsafe unpacked study path.")
        source = tree.joinpath(*parts)
        if not source.resolve().is_relative_to(tree.resolve()):
            raise RuntimeError("Unpacked study path escapes its root.")
        data = source.read_bytes()
        total_bytes += len(data)
        if total_bytes > 512 * 1024 * 1024:
            raise RuntimeError("Unpacked study exceeds the safe copy size limit.")
        if (not isinstance(binding, dict) or type(binding.get("bytes")) is not int
                or len(data) != binding["bytes"] or hashlib.sha256(data).hexdigest() != binding.get("sha256")):
            raise RuntimeError(f"Unpacked study source binding failed: {relative}")
        payload[relative] = data
    if destination.exists():
        raise RuntimeError("Study extraction already exists. Restart the kernel/session rather than replacing frozen files.")
    copied = destination / "conseqax-kaggle-study"
    copied.mkdir(parents=True, exist_ok=False)
    for relative, data in {**payload, "bundle-manifest.json": manifest_bytes}.items():
        target = copied.joinpath(*PurePosixPath(relative).parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
    return copied

def run_command(arguments):
    command = [sys.executable, "-m", *map(str, arguments)]
    COMMAND_PLAN.append(command)
    if DRY_RUN:
        print("DRY RUN ·", " ".join(command))
        return
    environment = dict(os.environ)
    source_paths = [str(STUDY_DIR), str(Path(STUDY_DIR) / "native")]
    if environment.get("PYTHONPATH"):
        source_paths.append(environment["PYTHONPATH"])
    environment.update(PYTHONPATH=os.pathsep.join(source_paths), PYTHONUNBUFFERED="1", PYTHONUTF8="1")
    diagnostic_dir = Path(OUTPUT_DIR).parent / "conseqax-diagnostics"
    diagnostic_dir.mkdir(parents=True, exist_ok=True)
    label = re.sub(r"[^a-zA-Z0-9_.-]", "_", str(arguments[0]))
    log_path = diagnostic_dir / f"{len(COMMAND_PLAN):02d}-{label}-{time.time_ns()}.log"
    tail = deque(maxlen=40)
    print("Command output is also saved to:", log_path, flush=True)
    with log_path.open("x", encoding="utf-8", newline="\n") as log:
        with subprocess.Popen(command, cwd=STUDY_DIR, env=environment,
                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, encoding="utf-8", errors="replace", bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                    tail.append(line)
                returncode = process.wait()
            except BaseException:
                # Match subprocess.run's interruption behavior; do not leave the
                # collector running after its notebook command was interrupted.
                if process.poll() is None:
                    process.kill()
                process.wait()
                raise
    if returncode:
        print(f"Command stopped with exit code {returncode}. Full error log: {log_path}", flush=True)
        print("Last command output (the underlying error is below):", flush=True)
        print("".join(tail), end="", flush=True)
        print("Preserve the complete saved output. No collection or model request was automatically retried.", flush=True)
        raise subprocess.CalledProcessError(returncode, command, output="".join(tail))

require_pins()
BUNDLE_MODE, BUNDLE = discover_bundle()
PREVIOUS_CHECKPOINT = discover_resume()
if DRY_RUN:
    STUDY_DIR = TEMP_ROOT / "conseqax-frozen-source" / "conseqax-kaggle-study"
    print("OFFLINE TEST: GPU, extraction, downloads, installation, and model calls are skipped.")
else:
    if not Path("/kaggle").is_dir():
        raise RuntimeError("Run this released notebook in a Kaggle GPU session.")
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         check=True, capture_output=True, text=True)
    if not gpu.stdout.strip():
        raise RuntimeError("No GPU detected. Enable GPU in Kaggle Notebook settings.")
    print("GPU detected:", gpu.stdout.strip())
    opener = open_bundle if BUNDLE_MODE == "zip" else open_unpacked_bundle
    STUDY_DIR = opener(BUNDLE, TEMP_ROOT / "conseqax-frozen-source")
run_command(["research.kaggle_limbo.bundle", "verify", "--study-dir", STUDY_DIR])
print("Frozen study:", EXPECTED_SOURCE_COMMIT)
print("Previous checkpoint:", PREVIOUS_CHECKPOINT or "None — first session")
print("Output:", OUTPUT_DIR)



if time.monotonic() - _SESSION_STARTED_AT >= float(SESSION_HOURS) * 3600:
    raise RuntimeError("The study deadline has been reached. Preserve any outputs and start a fresh Kaggle session; do not reset this session's timer.")
try:
    run_command(["research.kaggle_limbo.setup", "--study-dir", STUDY_DIR,
                 "--runtime-dir", TEMP_ROOT / "conseqax-runtime",
                 "--models-dir", TEMP_ROOT / "conseqax-models"])
except subprocess.CalledProcessError:
    print("Setup failed; the original error is above. This setup cell does not start benchmark slots.")
    print("Setup-only downloads already make bounded recovery attempts. Preserve outputs if setup fails; launch this complete notebook in a fresh session rather than applying isolated repair cells.")
    print("If the budget is exhausted, use a fresh Kaggle session. Preserve and attach the entire latest saved output if any earlier collection started. Do not reset the timer or edit integrity pins.")
    raise
print("Runtime setup finished. Continue only if this cell completed successfully.")



# The runner reads the first-cell timestamp from the inherited environment.
# Pass the original 7-hour window: subtracting setup here would count it twice.
elapsed_seconds = time.monotonic() - _SESSION_STARTED_AT
remaining_hours = float(SESSION_HOURS) - elapsed_seconds / 3600
if remaining_hours <= 0:
    raise RuntimeError("The study deadline has been reached. Export and save; continue in a new session.")
collection_args = ["research.kaggle_limbo.run", "--study-dir", STUDY_DIR,
                   "--output", OUTPUT_DIR, "--session-hours", str(SESSION_HOURS)]
if PREVIOUS_CHECKPOINT is not None:
    collection_args.extend(["--resume-from", PREVIOUS_CHECKPOINT])
print(f"Time remaining before study deadline: {remaining_hours:.2f} hours. Planned matrix: 3,456 slots.")
run_command(collection_args)



run_command(["research.kaggle_limbo.run", "--study-dir", STUDY_DIR,
             "--output", OUTPUT_DIR, "--export-only"])
print("Confirm this background version saved its complete output. Interactive runs: Quick Save with outputs; do not start a fresh Save & Run All to save results.")
print("Portable continuation:", Path(OUTPUT_DIR).parent / "checkpoint.zip")
print("No result or publication claim is made by this export.")